# AI Sudoku Game — Predicate Calculus + Z3
A playable Sudoku game for Google Colab. `Val(r,c,d)` is true when row `r`, column `c` contains digit `d`.

**Play:** Select a blank cell, use the number pad, and press Check, Hint, Solve with AI, or New Game. Run every cell from top to bottom.

In [ ]:
!pip -q install z3-solver

## 1. AI solver: Boolean predicates and Sudoku constraints

In [ ]:
from z3 import Bool, Solver, Or, Not, sat, is_true
import random, json
from IPython.display import display, HTML, Javascript
from google.colab import output

DIGITS = range(1, 10)

def solve_sudoku(puzzle):
    if len(puzzle) != 9 or any(len(row) != 9 for row in puzzle):
        raise ValueError("Puzzle must be a 9x9 grid.")
    if any(not isinstance(v, int) or v < 0 or v > 9 for row in puzzle for v in row):
        raise ValueError("Cells must be integers from 0 to 9.")

    solver = Solver()
    Val = {(r,c,d): Bool(f"Val_{r}_{c}_{d}")
           for r in DIGITS for c in DIGITS for d in DIGITS}

    # Every cell contains at least one digit.
    for r in DIGITS:
        for c in DIGITS:
            solver.add(Or([Val[r,c,d] for d in DIGITS]))

    # Every cell contains at most one digit.
    for r in DIGITS:
        for c in DIGITS:
            for d1 in DIGITS:
                for d2 in range(d1+1, 10):
                    solver.add(Or(Not(Val[r,c,d1]), Not(Val[r,c,d2])))

    # Row constraints: a digit cannot occur in two columns of one row.
    for r in DIGITS:
        for d in DIGITS:
            for c1 in DIGITS:
                for c2 in range(c1+1, 10):
                    solver.add(Or(Not(Val[r,c1,d]), Not(Val[r,c2,d])))

    # Column constraints: a digit cannot occur in two rows of one column.
    for c in DIGITS:
        for d in DIGITS:
            for r1 in DIGITS:
                for r2 in range(r1+1, 10):
                    solver.add(Or(Not(Val[r1,c,d]), Not(Val[r2,c,d])))

    # 3x3 box constraints.
    for d in DIGITS:
        for br in (1,4,7):
            for bc in (1,4,7):
                box = [(r,c) for r in range(br,br+3) for c in range(bc,bc+3)]
                for i in range(9):
                    for j in range(i+1,9):
                        r1,c1 = box[i]; r2,c2 = box[j]
                        solver.add(Or(Not(Val[r1,c1,d]), Not(Val[r2,c2,d])))

    # Given clues are fixed facts.
    for r in range(9):
        for c in range(9):
            if puzzle[r][c]:
                solver.add(Val[r+1,c+1,puzzle[r][c]])

    if solver.check() != sat:
        return None
    model = solver.model()
    return [[next(d for d in DIGITS
                  if is_true(model.eval(Val[r,c,d], model_completion=True)))
             for c in DIGITS] for r in DIGITS]

PUZZLES = [
 [[5,3,0,0,7,0,0,0,0],[6,0,0,1,9,5,0,0,0],[0,9,8,0,0,0,0,6,0],
  [8,0,0,0,6,0,0,0,3],[4,0,0,8,0,3,0,0,1],[7,0,0,0,2,0,0,0,6],
  [0,6,0,0,0,0,2,8,0],[0,0,0,4,1,9,0,0,5],[0,0,0,0,8,0,0,7,9]],
 [[0,0,0,2,6,0,7,0,1],[6,8,0,0,7,0,0,9,0],[1,9,0,0,0,4,5,0,0],
  [8,2,0,1,0,0,0,4,0],[0,0,4,6,0,2,9,0,0],[0,5,0,0,0,3,0,2,8],
  [0,0,9,3,0,0,0,7,4],[0,4,0,0,5,0,0,3,6],[7,0,3,0,1,8,0,0,0]]
]
game = {"idx":0, "puzzle":[row[:] for row in PUZZLES[0]],
        "board":[row[:] for row in PUZZLES[0]], "hints":[],
        "solution":solve_sudoku(PUZZLES[0]),
        "message":"Welcome! Select an empty cell and enter a number."}
print("Z3 AI ready. The game interface is in the next cell.")

## 2. Interactive game board

In [ ]:
display(HTML(r"""
<div id="sudoku-app" style="font-family:Arial;max-width:520px;margin:10px auto;color:#17324d">
 <div style="background:#183b56;color:white;text-align:center;padding:16px;border-radius:14px">
  <h2 style="margin:0">AI SUDOKU</h2><div>Predicate Calculus · Boolean Constraints · Z3</div>
 </div>
 <div id="sudoku-board" style="display:grid;grid-template-columns:repeat(9,minmax(0,1fr));gap:1px;background:#183b56;border:2px solid #183b56;width:min(94vw,430px);aspect-ratio:1;margin:14px auto"></div>
 <div id="sudoku-pad" style="display:grid;grid-template-columns:repeat(10,1fr);gap:4px;width:min(94vw,430px);margin:auto"></div>
 <div style="display:flex;flex-wrap:wrap;justify-content:center;gap:6px;margin:12px">
  <button onclick="sudokuAction('new')">New Game</button>
  <button onclick="sudokuAction('check')">Check</button>
  <button onclick="sudokuAction('hint')">Hint</button>
  <button onclick="sudokuAction('solve')" style="background:#176b62;color:white">Solve with AI</button>
  <button onclick="sudokuAction('clear')">Clear entries</button>
 </div>
 <div id="sudoku-status" style="background:#f1f6f8;border-radius:10px;padding:10px;text-align:center;min-height:24px;margin:8px">Loading…</div>
 <p style="font-size:12px;color:#5c6f82;text-align:center">Choose a blank cell, then use the number pad. Original clues are locked.</p>
</div>
"""))
display(Javascript(r"""
(function(){
 if(window.sudokuReady) return; window.sudokuReady=true;
 window.sudokuSelected=[0,0];
 window.sudokuSend=(action,r,c,value)=>google.colab.kernel.invokeFunction('sudoku.handle',[{action,r,c,value}],{});
 window.sudokuAction=(action)=>window.sudokuSend(action,...window.sudokuSelected,0);
 window.sudokuSelect=(r,c)=>{window.sudokuSelected=[r,c];window.sudokuSend('select',r,c,0)};
 window.sudokuNumber=(n)=>window.sudokuSend('input',...window.sudokuSelected,n);
 const board=document.getElementById('sudoku-board'), pad=document.getElementById('sudoku-pad');
 for(let r=0;r<9;r++) for(let c=0;c<9;c++){
  let b=document.createElement('button'); b.id=`sudoku-${r}-${c}`; b.className='sudoku-cell';
  b.style.cssText='border:0;background:white;color:#17324d;font-weight:bold;font-size:clamp(15px,4vw,22px);padding:0;min-width:0';
  if(c===2||c===5)b.style.borderRight='2px solid #183b56';
  if(r===2||r===5)b.style.borderBottom='2px solid #183b56';
  b.onclick=()=>window.sudokuSelect(r,c); board.appendChild(b);
 }
 for(let n=1;n<=9;n++){let b=document.createElement('button');b.textContent=n;b.onclick=()=>window.sudokuNumber(n);pad.appendChild(b)}
 let er=document.createElement('button');er.textContent='⌫';er.onclick=()=>window.sudokuNumber(0);pad.appendChild(er);
 window.sudokuRender=(s)=>{
  for(let r=0;r<9;r++)for(let c=0;c<9;c++){
   let b=document.getElementById(`sudoku-${r}-${c}`);b.textContent=s.board[r][c]||'';
   b.disabled=!!s.givens[r][c];b.style.background=s.givens[r][c]?'#eaf0f6':'white';
   b.style.color=s.givens[r][c]?'#17324d':'#087f5b';
   if(s.errors.some(([x,y])=>x===r&&y===c)){b.style.background='#ffe0df';b.style.color='#a61b1b'}
   if(s.hints.some(([x,y])=>x===r&&y===c))b.style.color='#087f5b';
   if(s.selected[0]===r&&s.selected[1]===c)b.style.outline='3px solid #14806a';else b.style.outline='none';
  }
  document.getElementById('sudoku-status').textContent=s.message;
 };
})();
"""))

## 3. Game controls and AI feedback

In [ ]:
def duplicate_cells(board):
    bad=set()
    units=[]
    units += [[(r,c) for c in range(9)] for r in range(9)]
    units += [[(r,c) for r in range(9)] for c in range(9)]
    units += [[(r,c) for r in range(br,br+3) for c in range(bc,bc+3)]
              for br in (0,3,6) for bc in (0,3,6)]
    for unit in units:
        seen={}
        for r,c in unit:
            v=board[r][c]
            if v: seen.setdefault(v,[]).append((r,c))
        for group in seen.values():
            if len(group)>1: bad.update(group)
    return sorted(bad)

def render_game(selected=(0,0), errors=None):
    state={"board":game["board"],
           "givens":[[int(bool(v)) for v in row] for row in game["puzzle"]],
           "selected":list(selected),"errors":[list(x) for x in (errors or [])],
           "hints":game["hints"],"message":game["message"]}
    display(Javascript("window.sudokuRender("+json.dumps(state)+");"))

def new_game():
    game["idx"]=(game["idx"]+1)%len(PUZZLES)
    game["puzzle"]=[row[:] for row in PUZZLES[game["idx"]]]
    game["board"]=[row[:] for row in game["puzzle"]]
    game["solution"]=solve_sudoku(game["puzzle"])
    game["hints"]=[]
    game["message"]=f"Puzzle {game['idx']+1} loaded. Good luck!"

def handle_sudoku(payload):
    action=payload.get("action"); r=int(payload.get("r",0)); c=int(payload.get("c",0))
    value=int(payload.get("value",0)); selected=(r,c); errors=[]
    if action=="select":
        game["message"]=f"Selected row {r+1}, column {c+1}. Choose a digit."
    elif action=="input":
        if game["puzzle"][r][c]:
            game["message"]="Original clues are locked. Select an empty cell."
        else:
            game["board"][r][c]=value
            game["hints"]=[x for x in game["hints"] if x != [r,c]]
            game["message"]="Entry updated. Press Check to inspect your work."
    elif action=="new":
        new_game(); selected=(0,0)
    elif action=="clear":
        for rr in range(9):
            for cc in range(9):
                if not game["puzzle"][rr][cc]: game["board"][rr][cc]=0
        game["hints"]=[]
        game["message"]="Entries cleared. Original clues remain."
    elif action=="check":
        errors=duplicate_cells(game["board"])
        if errors:
            game["message"]=f"Rule conflicts found in {len(errors)} cells (highlighted red)."
        else:
            wrong=[(rr,cc) for rr in range(9) for cc in range(9)
                   if game["board"][rr][cc] and game["board"][rr][cc]!=game["solution"][rr][cc]]
            if wrong:
                errors=wrong
                game["message"]=f"{len(wrong)} filled cell(s) do not match the solution (red)."
            elif all(all(row) for row in game["board"]):
                game["message"]="Congratulations! You solved the Sudoku correctly!"
            else:
                game["message"]="No mistakes so far. Keep going!"
    elif action=="hint":
        empty=[(rr,cc) for rr in range(9) for cc in range(9) if not game["board"][rr][cc]]
        if empty:
            rr,cc=random.choice(empty); game["board"][rr][cc]=game["solution"][rr][cc]
            game["hints"].append([rr,cc]); selected=(rr,cc)
            game["message"]=f"AI hint: row {rr+1}, column {cc+1} is {game['solution'][rr][cc]}."
        else: game["message"]="There are no empty cells."
    elif action=="solve":
        solution=solve_sudoku(game["puzzle"])
        if solution is None: game["message"]="No solution exists for this puzzle."
        else:
            game["board"]=[row[:] for row in solution]; game["hints"]=[]
            game["message"]="AI solved it! Z3 found a satisfying assignment for Val(r,c,d)."
    render_game(selected,errors)

output.register_callback("sudoku.handle",handle_sudoku)
render_game()
print("Game controls connected. Play in the board above.")

## 4. Verify the AI solution

In [ ]:
def verify_solution(puzzle, solution):
    if solution is None: return False
    target=set(range(1,10))
    if any(set(row)!=target for row in solution): return False
    if any({solution[r][c] for r in range(9)}!=target for c in range(9)): return False
    for br in (0,3,6):
        for bc in (0,3,6):
            if {solution[r][c] for r in range(br,br+3) for c in range(bc,bc+3)}!=target:
                return False
    return all(not puzzle[r][c] or puzzle[r][c]==solution[r][c]
               for r in range(9) for c in range(9))

assert verify_solution(PUZZLES[0],solve_sudoku(PUZZLES[0]))
print("Test passed: rows, columns, boxes and original clues are valid.")

## Predicate-calculus summary
- `Val(r,c,d)`: the cell at row `r`, column `c` contains digit `d`.
- At least one digit per cell + at most one digit per cell = exactly one digit per cell.
- For a row, a digit cannot be true in two distinct columns: `¬Val(r,c1,d) ∨ ¬Val(r,c2,d)`.
- The same pattern applies to columns and to distinct cells in the same 3×3 box.
- Original clues are asserted as facts. Z3 searches for a satisfying assignment.

**Limitations:** This version has two built-in puzzles. It is designed to run in Google Colab because the interactive board uses Colab's JavaScript callback support.